# 🏏 IPL Data Analytics Project — 2008 to 2025
### Comprehensive EDA + Visualisation + Machine Learning
**Dataset:** 6-file relational IPL dataset | 1,169 matches | 278,205 deliveries | 18 seasons

**Tools:** Python · pandas · NumPy · Matplotlib · Seaborn · Scikit-learn

---

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn reportlab -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
import os
import json
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('✅ All libraries imported successfully!')

## 📁 Step 2: Upload Dataset Files
Upload all 6 CSV files from Kaggle: `all_ipl_matches_data.csv`, `all_ball_by_ball_data.csv`, 
`all_players-data-updated.csv`, `all_teams_data.csv`, `all_team_aliases.csv`, `IPL_finals.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('✅ Files uploaded:', list(uploaded.keys()))

## 🔧 Step 3: Load & Merge Data

In [ ]:
# Load all 6 files
matches  = pd.read_csv('all_ipl_matches_data.csv')
deliv    = pd.read_csv('all_ball_by_ball_data.csv')
teams    = pd.read_csv('all_teams_data.csv')
players  = pd.read_csv('all_players-data-updated.csv')
aliases  = pd.read_csv('all_team_aliases.csv')
finals   = pd.read_csv('IPL_finals.csv')

# Map team_id → team_name
t_map = dict(zip(teams['team_id'], teams['team_name']))
for col in ['team1','team2','toss_winner','match_winner']:
    matches[col] = matches[col].map(t_map)
for col in ['team_batting','team_bowling']:
    deliv[col] = deliv[col].map(t_map)

# Map player_id → player_name
p_map = dict(zip(players['player_id'], players['player_name']))
matches['pom_name'] = matches['player_of_match'].map(p_map)

# Normalise season
matches['season'] = matches['season'].astype(str).str.replace('/21','').astype(int)
deliv['season_id'] = deliv['season_id'].astype(str).str.replace('/21','').astype(int)

valid = matches[matches['result']=='win'].copy()

print(f'Matches: {len(matches)} | Deliveries: {len(deliv)} | Seasons: {matches["season"].nunique()}')
print(f'Teams: {matches["team1"].nunique()}')
matches.head()

## 🔍 Step 4: Data Quality Check

In [ ]:
print('=== MATCHES - Missing Values ===')
print(matches.isnull().sum()[matches.isnull().sum()>0])

print('\n=== DELIVERIES - Missing Values ===')
print(deliv.isnull().sum()[deliv.isnull().sum()>0])

print('\n=== MATCHES - Basic Stats ===')
print(matches[['win_by_runs','win_by_wickets']].describe())

print('\n=== Result Types ===')
print(matches['result'].value_counts())

## 📊 Step 5: Exploratory Data Analysis
### 5.1 Most Winning Teams

In [ ]:
# Dark theme setup
BG='#0d1117'; CARD='#161b22'; ACC1='#f97316'; ACC2='#3b82f6'
ACC3='#22c55e'; ACC4='#a855f7'; TEXT='#e6edf3'; MUTED='#8b949e'
TEAM_COLORS = {
    'Mumbai Indians':'#004BA0','Chennai Super Kings':'#F4C430',
    'Kolkata Knight Riders':'#3A225D','Royal Challengers Bangalore':'#EC1C24',
    'Sunrisers Hyderabad':'#FF822A','Punjab Kings':'#AA4545',
    'Delhi Capitals':'#0078BC','Rajasthan Royals':'#2D68C4',
    'Gujarat Titans':'#1C4B9C','Lucknow Super Giants':'#00A86B',
}

def style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT); ax.yaxis.label.set_color(TEXT)
    for sp in ax.spines.values(): sp.set_color('#30363d')
    if title:  ax.set_title(title, color=TEXT, fontsize=13, fontweight='bold')
    if xlabel: ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    ax.grid(axis='y', color='#30363d', linewidth=0.5, alpha=0.6)
    ax.set_axisbelow(True)

wins = valid['match_winner'].value_counts().head(10)
colors = [TEAM_COLORS.get(t, ACC1) for t in wins.index]
fig, ax = plt.subplots(figsize=(11,5.5), facecolor=BG)
bars = ax.barh(wins.index[::-1], wins.values[::-1], color=colors[::-1], height=0.65, edgecolor='none')
for bar,val in zip(bars, wins.values[::-1]):
    ax.text(val+1, bar.get_y()+bar.get_height()/2, str(val), va='center', color=TEXT, fontsize=9, fontweight='bold')
style_ax(ax,'🏆 Most Winning IPL Teams (2008–2025)','Total Wins')
ax.set_xlim(0, wins.max()*1.15)
ax.grid(axis='x',color='#30363d',linewidth=0.5,alpha=0.6); ax.grid(axis='y',visible=False)
plt.tight_layout(); plt.savefig('01_most_wins.png',dpi=150,bbox_inches='tight',facecolor=BG); plt.show()

### 5.2 Season Trends

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,4.5), facecolor=BG)

sm = matches.groupby('season').size()
axes[0].fill_between(sm.index, sm.values, alpha=0.15, color=ACC2)
axes[0].plot(sm.index, sm.values, color=ACC2, marker='o', linewidth=2.5, markersize=7)
for x,y in zip(sm.index, sm.values): axes[0].text(x, y+0.5, str(y), ha='center', color=TEXT, fontsize=7)
style_ax(axes[0],'📅 Matches Per Season','Season','Matches')
axes[0].set_xticks(sm.index); axes[0].set_xticklabels(sm.index, rotation=45, color=TEXT, fontsize=7)

sr = deliv.groupby('season_id')['total_runs'].sum()
axes[1].fill_between(sr.index, sr.values, alpha=0.15, color=ACC3)
axes[1].plot(sr.index, sr.values, color=ACC3, marker='s', linewidth=2.5, markersize=7)
for x,y in zip(sr.index, sr.values): axes[1].text(x, y+200, f'{y//1000}K', ha='center', color=TEXT, fontsize=7)
style_ax(axes[1],'📈 Total Runs Per Season','Season','Total Runs')
axes[1].set_xticks(sr.index); axes[1].set_xticklabels(sr.index, rotation=45, color=TEXT, fontsize=7)

plt.tight_layout(); plt.savefig('02_season_trends.png',dpi=150,bbox_inches='tight',facecolor=BG); plt.show()

### 5.3 Toss Analysis

In [ ]:
v = valid.copy()
v['toss_match_win'] = v['toss_winner'] == v['match_winner']
pct = v['toss_match_win'].mean()*100
print(f'Toss win → match win: {pct:.1f}%')
print(f'Toss Decision breakdown:')
print(matches['toss_decision'].value_counts())

fig,axes = plt.subplots(1,2, figsize=(11,4.5), facecolor=BG)
sizes=[pct,100-pct]
wedges,texts,at = axes[0].pie(sizes,labels=['Won Match','Lost Match'],colors=[ACC1,CARD],
    autopct='%1.1f%%',startangle=90,textprops={'color':TEXT},wedgeprops={'edgecolor':BG,'linewidth':2})
for a in at: a.set_color(BG); a.set_fontweight('bold')
axes[0].set_facecolor(BG); axes[0].set_title('Toss Win → Match Win?',color=TEXT,fontsize=12,fontweight='bold')
td = matches['toss_decision'].value_counts()
axes[1].bar(td.index, td.values, color=[ACC2,ACC3], edgecolor='none', width=0.5)
for i,(idx,val) in enumerate(td.items()): axes[1].text(i, val+3, str(val), ha='center', color=TEXT, fontweight='bold')
style_ax(axes[1],'Toss Decision Distribution',ylabel='Count')
plt.tight_layout(); plt.savefig('03_toss.png',dpi=150,bbox_inches='tight',facecolor=BG); plt.show()

### 5.4 Top Batsmen & Bowlers

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5.5), facecolor=BG)

top_bat = deliv.groupby('batter')['batter_runs'].sum().sort_values(ascending=False).head(10)
gc = plt.cm.YlOrRd(np.linspace(0.4,0.9,10))
axes[0].bar(range(10), top_bat.values, color=gc, edgecolor='none', width=0.65)
for i,(v2) in enumerate(top_bat.values): axes[0].text(i, v2+30, f'{v2:,}', ha='center', color=TEXT, fontsize=7, fontweight='bold')
axes[0].set_xticks(range(10)); axes[0].set_xticklabels(top_bat.index, rotation=30, ha='right', color=TEXT, fontsize=8)
style_ax(axes[0],'🏏 Top 10 Run Scorers',ylabel='Total Runs')

wkts = deliv[deliv['is_wicket']==True].groupby('bowler')['is_wicket'].count().sort_values(ascending=False).head(10)
gc2 = plt.cm.PuBu(np.linspace(0.4,0.95,10))
axes[1].bar(range(10), wkts.values, color=gc2[::-1], edgecolor='none', width=0.65)
for i,v2 in enumerate(wkts.values): axes[1].text(i, v2+0.5, str(v2), ha='center', color=TEXT, fontsize=7, fontweight='bold')
axes[1].set_xticks(range(10)); axes[1].set_xticklabels(wkts.index, rotation=30, ha='right', color=TEXT, fontsize=8)
style_ax(axes[1],'🎯 Top 10 Wicket Takers',ylabel='Wickets')

plt.tight_layout(); plt.savefig('04_bat_bowl.png',dpi=150,bbox_inches='tight',facecolor=BG); plt.show()

### 5.5 Win% Heatmap — Top 6 Teams by Season

In [ ]:
top6 = valid['match_winner'].value_counts().head(6).index.tolist()
all_t = pd.concat([matches[['season','team1']].rename(columns={'team1':'team'}),
                   matches[['season','team2']].rename(columns={'team2':'team'})])
played = all_t.groupby(['season','team']).size().reset_index(name='played')
won_df = valid.groupby(['season','match_winner']).size().reset_index(name='won')
won_df.columns = ['season','team','won']
perf = played.merge(won_df, on=['season','team'], how='left').fillna(0)
perf['win_pct'] = (perf['won']/perf['played']*100).round(1)
perf6 = perf[perf['team'].isin(top6)]
pivot = perf6.pivot(index='team', columns='season', values='win_pct').fillna(0)

fig,ax = plt.subplots(figsize=(14,5), facecolor=BG)
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
    linewidths=0.3, linecolor=BG, cbar_kws={'shrink':0.7}, annot_kws={'size':7,'color':'black'})
ax.set_facecolor('#161b22')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', color='#e6edf3')
plt.setp(ax.get_yticklabels(), rotation=0, color='#e6edf3')
ax.set_title('🔥 Win% Heatmap — Top 6 Teams by Season', color='#e6edf3', fontsize=13, fontweight='bold')
ax.set_xlabel('Season', color='#8b949e'); ax.set_ylabel('', color='#8b949e')
plt.tight_layout(); plt.savefig('05_heatmap.png',dpi=150,bbox_inches='tight',facecolor=BG); plt.show()

## 🤖 Step 6: Machine Learning — Match Outcome Prediction

In [ ]:
df = matches[matches['result']=='win'].copy().dropna(subset=['match_winner','team1','team2'])

# Feature: historical win rate
win_rate = {}
for team in df['team1'].dropna().unique():
    played = len(df[(df['team1']==team)|(df['team2']==team)])
    won = len(df[df['match_winner']==team])
    win_rate[team] = won/played if played>0 else 0.5

df['team1_wr'] = df['team1'].map(win_rate).fillna(0.5)
df['team2_wr'] = df['team2'].map(win_rate).fillna(0.5)
df['toss_home'] = (df['toss_winner']==df['team1']).astype(int)
df['field_first'] = (df['toss_decision']=='field').astype(int)

le_team = LabelEncoder()
le_team.fit(pd.concat([df['team1'],df['team2'],df['match_winner']]).dropna().unique())
df['t1_enc'] = le_team.transform(df['team1'])
df['t2_enc'] = le_team.transform(df['team2'])
le_v = LabelEncoder()
df['venue_enc'] = le_v.fit_transform(df['venue'].fillna('Unknown'))
df['winner_enc'] = le_team.transform(df['match_winner'])

features = ['t1_enc','t2_enc','team1_wr','team2_wr','toss_home','field_first','venue_enc','season']
X = df[features]; y = df['winner_enc']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# Train & Compare Models
models = {
    'Random Forest':  RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42),
    'Gradient Boost': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
    'Logistic Reg':   LogisticRegression(max_iter=1000, random_state=42),
}
results = {}
for name,model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc  = accuracy_score(y_test, pred)
    cv   = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
    results[name] = {'acc':acc,'cv':cv,'model':model,'pred':pred}
    print(f'{name}: Test Acc={acc:.3f} | CV Acc={cv:.3f}')

best_name = max(results, key=lambda k: results[k]['cv'])
print(f'\n✅ Best Model: {best_name} (CV={results[best_name]["cv"]:.3f})')

In [ ]:
# Feature Importance Plot
rf = results['Random Forest']['model']
fi = pd.Series(rf.feature_importances_, index=features).sort_values()
fig,ax = plt.subplots(figsize=(9,4.5), facecolor=BG)
cols = plt.cm.YlOrRd(np.linspace(0.3,0.9,len(fi)))
ax.barh(fi.index, fi.values, color=cols, edgecolor='none', height=0.6)
for i,(idx,val) in enumerate(fi.items()): ax.text(val+0.002, i, f'{val:.3f}', va='center', color=TEXT, fontsize=8)
style_ax(ax,'🔍 Feature Importance — Random Forest','Importance Score')
ax.grid(axis='x',color='#30363d',linewidth=0.5,alpha=0.6); ax.grid(axis='y',visible=False)
plt.tight_layout(); plt.show()

## 📥 Step 7: Download Results

In [ ]:
from google.colab import files

# Download all saved charts
chart_files = ['01_most_wins.png','02_season_trends.png','03_toss.png','04_bat_bowl.png','05_heatmap.png']
for f in chart_files:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')
print('✅ All files downloaded!')

---
## 📋 Summary of Findings

| Metric | Value |
|---|---|
| Total Matches Analysed | 1,169 |
| Total Deliveries | 278,205 |
| Seasons Covered | 2008–2025 (18 seasons) |
| Most Successful Team | Mumbai Indians (151 wins) |
| Top Run Scorer | V Kohli (8,671 runs) |
| Top Wicket Taker | YS Chahal (229 wickets) |
| Toss Win Rate | 51.6% (near-random) |
| Best ML Model | Random Forest (~50% CV) |

---
**Note on ML Accuracy:** 50% CV accuracy in a 14-class prediction problem (14 teams) is meaningful — 
random chance would be ~7%. The model successfully identifies that team quality (win rate) 
is the primary predictor, which aligns with domain knowledge.

---
*Project by HARENDER CHHOKER| IPL Analytics 2008–2025 | Tools: Python, pandas, scikit-learn*